# Text Summarization with Pretrained LLMs

This notebook explores how different pretrained Large Language Model architectures behave on text generation and summarization tasks, using the Hugging Face `transformers` library.

Three models are compared, each representative of a different LLM design:

1. **BERT2BERT** (`patrickvonplaten/bert2bert_cnn_daily_mail`) — an encoder-decoder model built from two BERT stacks and fine-tuned specifically for summarization on the CNN/DailyMail dataset.
2. **FLAN-T5** (`google/flan-t5-small`) — an instruction-tuned encoder-decoder model that can perform many tasks (including summarization) simply by describing the task in the prompt.
3. **GPT-2** — a decoder-only, purely autoregressive language model with no instruction tuning, used here to illustrate plain next-token prediction (greedy decoding implemented manually).

For each model we load the pretrained weights, inspect the architecture, and run inference on a sample news-style article to observe how well it summarizes (or, in GPT-2's case, simply continues) the text.

In [36]:
from transformers import AutoTokenizer, EncoderDecoderModel
import torch

## 1. BERT2BERT: an encoder-decoder model fine-tuned for summarization

We load `patrickvonplaten/bert2bert_cnn_daily_mail`, a `EncoderDecoderModel` composed of two BERT stacks: a BERT encoder that reads the source article, and a BERT decoder (with cross-attention added) that generates the summary token by token. Unlike GPT-2 below, this model was specifically fine-tuned on the CNN/DailyMail summarization dataset, so it is expected to produce a genuine abstractive summary rather than just continuing the text.

The model is set to `eval()` mode (disabling dropout) and its architecture is printed to show the encoder/decoder BERT stacks and the cross-attention layers that let the decoder attend to the encoder's output.

In [37]:
model = EncoderDecoderModel.from_pretrained("patrickvonplaten/bert2bert_cnn_daily_mail")
tokenizer = AutoTokenizer.from_pretrained("patrickvonplaten/bert2bert_cnn_daily_mail")

model.eval()
model

Loading weights:   0%|          | 0/523 [00:00<?, ?it/s]

EncoderDecoderModel(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [38]:
ARTICLE_TO_SUMMARIZE = (
    "Scientists have discovered a new species of deep-sea fish living near hydrothermal vents in the Pacific Ocean. "
    "The fish, characterized by its translucent skin and bioluminescent organs, thrives in extreme pressure and temperature conditions. "
    "Researchers believe this discovery could provide insights into how life adapts to hostile environments. "
    "The species was found during a submarine expedition using remotely operated vehicles equipped with high-resolution cameras. "
    "Further studies are planned to understand its role in the deep-sea ecosystem and its potential applications in biotechnology."
)

A sample news-style article (`ARTICLE_TO_SUMMARIZE`) is defined and used as the common input for all three models in this notebook, so their outputs can be compared fairly.

The article is tokenized into `input_ids`, fed to `model.generate()` to produce the summary token IDs, and finally decoded back into readable text with the tokenizer.

In [39]:
input_ids = tokenizer(ARTICLE_TO_SUMMARIZE, return_tensors='pt').input_ids
generated_ids = model.generate(input_ids)
print(generated_ids)

tensor([[  101,  6529,  2031,  3603,  1037,  2047,  2427,  1997,  2784,  1011,
          2712,  3869,  2542,  2379, 18479, 23367, 28287,  1999,  1996,  3534,
          4153,  1012,  1996,  3869, 25220,  2015,  1999,  6034,  3778,  1998,
          4860,  3785,  1012,  6529,  2903,  2023,  5456,  2071,  3073, 20062,
          2046,  2129,  2166, 15581,  2015,  2000, 10420, 10058,  1012,  2582,
          2913,  2024,  3740,  2000,  3305,  2049,  2535,  1999,  1996, 16012,
         23874,  1998,  2049,  4022,  1999, 20353,  1012,   102]])


In [40]:
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)

scientists have discovered a new species of deep - sea fish living near hydrothermal vents in the pacific ocean. the fish thrives in extreme pressure and temperature conditions. scientists believe this discovery could provide insights into how life adapts to hostile environments. further studies are planned to understand its role in the biosphere and its potential in biotechnology.


## 2. FLAN-T5: an instruction-tuned encoder-decoder model

Next we switch to `google/flan-t5-small`, a `T5ForConditionalGeneration` model instruction-tuned on a mixture of tasks. Unlike BERT2BERT, which is a summarizer by construction, FLAN-T5 is a general-purpose text-to-text model: what it does depends entirely on how the input prompt is phrased.

The architecture is printed to show T5's relative-attention encoder stack and its decoder stack with added cross-attention to the encoder.

In [41]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

model.eval()
model

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

### Generic prompt (no explicit task instruction)

Before testing summarization, we first prompt FLAN-T5 with an open-ended sentence-completion instruction ("Continue the sentence: ..."). This checks the model's general text-generation behavior with `generate()` (greedy decoding, `max_length=50`) before we ask it to perform the summarization task specifically.

In [42]:
input_ids = tokenizer.encode("Continue the sentece: I was in Paris and decided that", return_tensors='pt')
greedy_output = model.generate(input_ids, max_length=50)

print(greedy_output)

tensor([[   0, 1919,   47,    3,    9,  207,  286,   12, 1049,    5,    1]])


In [43]:
greedy_text = tokenizer.decode(greedy_output[0], skip_special_tokens=True)
print(greedy_text)

Paris was a good place to stay.


### Summarization prompt

Now the same `ARTICLE_TO_SUMMARIZE` used for BERT2BERT is prefixed with the explicit instruction `"Summarize: "` and passed to FLAN-T5. Because the model is instruction-tuned, this simple prompt change is enough to steer it toward producing a summary instead of a free continuation — allowing a direct comparison with BERT2BERT's dedicated summarization output.

In [44]:
input_ids = tokenizer.encode(f"Summarize: {ARTICLE_TO_SUMMARIZE}", return_tensors='pt')
output = model.generate(input_ids, max_length=50)
text = tokenizer.decode(output[0], skip_special_tokens=True)

print(text)

Scientists have discovered a new species of deep-sea fish that thrives in extreme pressure and temperature conditions.


## 3. GPT-2: a decoder-only autoregressive model

Finally, we load plain `gpt2` — a decoder-only Transformer with no encoder-decoder structure and no instruction tuning. GPT-2 was trained purely to predict the next token given the preceding context, so it has no built-in notion of "summarize"; it can only continue whatever text it is given.

To make this explicit, instead of calling the high-level `generate()` API used for the previous two models, we implement **greedy decoding manually**: at each step we run a forward pass, take the `argmax` of the logits for the last position to pick the most likely next token, append it to the sequence, and repeat. This illustrates exactly how autoregressive generation works under the hood.

In [45]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

model.eval()
model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [46]:
start = "Natural Language Processing is slowly becoming"
indexed_tokens = tokenizer.encode(start)
for i in range(75):
    tokens_tensor = torch.tensor([indexed_tokens])
    with torch.no_grad():
        outputs = model(tokens_tensor)
        predictions = outputs.logits
        predicted_index = torch.argmax(predictions[0, -1, :]).item()
        indexed_tokens = indexed_tokens + [predicted_index]

predicted_text = tokenizer.decode(indexed_tokens)
print(predicted_text)

Natural Language Processing is slowly becoming a reality.

The first step is to create a language processing system that can be used to create a language. This is done by using a language processing system that is built on top of a language processing system.

The language processing system is a set of tools that can be used to create a language. The language processing system is a set of tools that can
